# Задание #2. Классификация ландшафтов Марса

Автономные марсоходы, такие как [Curiosity](https://en.wikipedia.org/wiki/Curiosity_(rover)) и [Perseverance](https://en.wikipedia.org/wiki/Perseverance_(rover)), ежедневно отправляют на Землю тысячи снимков поверхности Марса. Для планирования безопасного маршрута и поиска научно значимых объектов (например, следов древних рек или необычных минералов) бортовой компьютер должен мгновенно понимать, что находится перед ним.

Вы — разработчик системы компьютерного зрения для нового поколения марсоходов. Ваша задача — создать нейросеть, способную по черно-белым снимкам с камер навигации безошибочно определять тип поверхности. Ошибка в классификации может привести к тому, что ровер застрянет в зыбучих песках или повредит колеса об острые скалы.

## Данные
Имеется набор из `29,718` черно-белых снимков поверхности Марса, полученных телескопом [HiRISE](https://en.wikipedia.org/wiki/HiRISE). Снимки поделены на 15 классов:

<table>
    <thead>
        <tr>
            <th align="left">ID класса</th>
            <th align="left">Имя класса</th>
            <th align="left">Описание</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td align="left">0</td>
            <td align="left">ael</td>
            <td align="left">Aeolian Straight (Эоловы плоскости)</td>
        </tr>
        <tr>
            <td align="left">1</td>
            <td align="left">rou</td>
            <td align="left">Rough Terrain (Пересеченная местность)</td>
        </tr>
        <tr>
            <td align="left">2</td>
            <td align="left">cli</td>
            <td align="left">Cliff (Утес)</td>
        </tr>
        <tr>
            <td align="left">3</td>
            <td align="left">aec</td>
            <td align="left">Aeolian Curved (Эоловы неровности)</td>
        </tr>
        <tr>
            <td align="left">4</td>
            <td align="left">tex</td>
            <td align="left">Textured Terrain (Неровная поверхность)</td>
        </tr>
        <tr>
            <td align="left">5</td>
            <td align="left">smo</td>
            <td align="left">Smooth Terrain (Гладкая поверхность)</td>
        </tr>
        <tr>
            <td align="left">6</td>
            <td align="left">fss</td>
            <td align="left">Mass Wasting (Последствия оползней)</td>
        </tr>
        <tr>
            <td align="left">7</td>
            <td align="left">rid</td>
            <td align="left">Ridge (Гряда)</td>
        </tr>
        <tr>
            <td align="left">8</td>
            <td align="left">fse</td>
            <td align="left">Slope Streaks (Полосы на склонах)</td>
        </tr>
        <tr>
            <td align="left">9</td>
            <td align="left">sfe</td>
            <td align="left">Mounds (Холмы)</td>
        </tr>
        <tr>
            <td align="left">10</td>
            <td align="left">fsf</td>
            <td align="left">Channel (Каналы)</td>
        </tr>
        <tr>
            <td align="left">11</td>
            <td align="left">fsg</td>
            <td align="left">Gullies (Овраги)</td>
        </tr>
        <tr>
            <td align="left">12</td>
            <td align="left">sfx</td>
            <td align="left">Crater Field (Поле мелких кратеров)</td>
        </tr>
        <tr>
            <td align="left">13</td>
            <td align="left">cra</td>
            <td align="left">Crater (Большой кратер)</td>
        </tr>
        <tr>
            <td align="left">14</td>
            <td align="left">mix</td>
            <td align="left">Mixed Terrain (Смешенная поверхность)</td>
        </tr>
    </tbody>
</table>

Пример изображений:
</br>
</br>
*aec*
</br>
![aec](https://i.imgur.com/NeI4hkj.jpeg)
</br>
</br>
*sfx*
</br>
![sfx](https://i.imgur.com/QQ7MG5D.jpeg)
</br>
</br>
*sfe*
</br>
![sfe](https://i.imgur.com/Y6LFAhs.jpeg)
</br>
### Структура данных
Дано два поля:
- `image`: черно-белое изображение поверхности Марса `PIL.Image.Image`, исходный размер — `200x200` пикселя.
- `label`: целочисленное значение, содержащее номер класса `(0–14)`

Набор делится на:
- тренировочные данные: `~11.3k` изображений
- тестовые данные: `1.61k` изображений
- валидационные данные: `3.23k`
- выборки для few shot обучения пакетами по `15` изображений, выборки `x1`, `x2`, `x5`, `x10`, `x15` и `x20`
- остаток данных размечен на портиции 

### Загрузка данных
Обратитесь к администрации и уточните, где можно загрузить набор, если нет доступа к ресурсу HuggingFace.

In [ ]:
from datasets import load_dataset

ds = load_dataset("Mirali33/mb-domars16k")

In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'few_shot_train_10_shot': 'data/few_shot_train_10_shot-00000-of-00001.parquet', 'few_shot_train_15_shot': 'data/few_shot_train_15_shot-00000-of-00001.parquet', 'few_shot_train_1_shot': 'data/few_shot_train_1_shot-00000-of-00001.parquet', 'few_shot_train_20_shot': 'data/few_shot_train_20_shot-00000-of-00001.parquet', 'few_shot_train_2_shot': 'data/few_shot_train_2_shot-00000-of-00001.parquet', 'few_shot_train_5_shot': 'data/few_shot_train_5_shot-00000-of-00001.parquet', 'partition_train_0.01x_partition': 'data/partition_train_0.01x_partition-00000-of-00001.parquet', 'partition_train_0.02x_partition': 'data/partition_train_0.02x_partition-00000-of-00001.parquet', 'partition_train_0.05x_partition': 'data/partition_train_0.05x_partition-00000-of-00001.parquet', 'partition_train_0.10x_partition': 'data/partition_train_0.10x_partition-00000-of-00001.parquet', 'partition_train_0.20x_partition': 'data/partition_train_0.20x_partition-00000-of-00001.parquet', 'partition_train_0.25x_partition': 'data/partition_train_0.25x_partition-00000-of-00001.parquet', 'partition_train_0.50x_partition': 'data/partition_train_0.50x_partition-00000-of-00001.parquet', 'val': 'data/val-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Mirali33/mb-domars16k/" + splits["train"])

In [ ]:
import polars as pl

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'few_shot_train_10_shot': 'data/few_shot_train_10_shot-00000-of-00001.parquet', 'few_shot_train_15_shot': 'data/few_shot_train_15_shot-00000-of-00001.parquet', 'few_shot_train_1_shot': 'data/few_shot_train_1_shot-00000-of-00001.parquet', 'few_shot_train_20_shot': 'data/few_shot_train_20_shot-00000-of-00001.parquet', 'few_shot_train_2_shot': 'data/few_shot_train_2_shot-00000-of-00001.parquet', 'few_shot_train_5_shot': 'data/few_shot_train_5_shot-00000-of-00001.parquet', 'partition_train_0.01x_partition': 'data/partition_train_0.01x_partition-00000-of-00001.parquet', 'partition_train_0.02x_partition': 'data/partition_train_0.02x_partition-00000-of-00001.parquet', 'partition_train_0.05x_partition': 'data/partition_train_0.05x_partition-00000-of-00001.parquet', 'partition_train_0.10x_partition': 'data/partition_train_0.10x_partition-00000-of-00001.parquet', 'partition_train_0.20x_partition': 'data/partition_train_0.20x_partition-00000-of-00001.parquet', 'partition_train_0.25x_partition': 'data/partition_train_0.25x_partition-00000-of-00001.parquet', 'partition_train_0.50x_partition': 'data/partition_train_0.50x_partition-00000-of-00001.parquet', 'val': 'data/val-00000-of-00001.parquet'}
df = pl.read_parquet('hf://datasets/Mirali33/mb-domars16k/' + splits['train'])


## Метрики
Так как в геологических выборках рамки между классами бывают довольно размытыми, точность работы модели следует считать не строгой Top-1 Accuracy, а по более мягкой Top-3 Accuracy.
В стандартном Top-1 Accuracy предсказание считается верным только в том случае, если класс с самой высокой вероятностью, который выдала модель, совпадает с реальной меткой. В Top-3 Accuracy предсказание считается верным, если реальная метка входит в число трех наиболее вероятных ответов модели.
Формально Top-3 Accuracy для выборки из N примеров считается следующим образом:
$$
Accuracy = \frac{1}{N} \sum_{i=1}^{N}[y_i \in top3(\hat{y}_i)]
$$
где:
- $y_i$ — истинный класс объекта.
- $y^i$ — вектор вероятностей всех классов, выданный моделью.
- $[func]$ — функция-индикатор (равна $1$, если условие верно, и $0$ в противном случае).

# Ограничение
Запрещено:
- дообучать моделы на набораx данных кроме представленного;
- обучать модели на тестовых наборах.

Работы, которые нарушили данные ограничения, **рассмотрены не будут**.

# Baseline
Ниже приведен базовый стенд для тестирования модели, обученной с помощью `PyTorch`

In [ ]:
import os
import time
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset

class MarsTerrainValidator:
    def __init__(self, model_path, test_limit=None):
        self.model_path = model_path
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.test_limit = test_limit
        
        self.img_size = 200 
        
        self.transform = transforms.Compose([
            transforms.Resize((self.img_size, self.img_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def load_test_data(self):
        ds = load_dataset("Mirali33/mb-domars16k", split="test")
        if self.test_limit:
            ds = ds.select(range(self.test_limit))
        return ds

    def calculate_topk_accuracy(self, output, target, k=3):
        """Расчет Top-K Accuracy"""
        with torch.no_grad():
            _, pred = output.topk(k, 1, True, True)
            pred = pred.t()
            correct = pred.eq(target.view(1, -1).expand_as(pred))
            res = correct[:k].reshape(-1).float().sum(0, keepdim=True)
            return res.item()

    def evaluate(self, model):
        test_ds = self.load_test_data()
        total_top1 = 0
        total_top3 = 0
        latencies = []
        n_samples = len(test_ds)

        model.eval()
        model.to(self.device)
        
        with torch.no_grad():
            for item in test_ds:
                image = item['image']
                label = torch.tensor([item['label']]).to(self.device)
                
                input_tensor = self.transform(image).unsqueeze(0).to(self.device)
                
                start_time = time.perf_counter()
                output = model(input_tensor)
                end_time = time.perf_counter()
                
                total_top1 += self.calculate_topk_accuracy(output, label, k=1)
                total_top3 += self.calculate_topk_accuracy(output, label, k=3)
                latencies.append(end_time - start_time)

        results = {
            "top1_accuracy": total_top1 / n_samples,
            "top3_accuracy": total_top3 / n_samples,
            "avg_latency_ms": np.mean(latencies) * 1000,
            "model_size_mb": os.path.getsize(self.model_path) / (1024 * 1024)
        }
        
        return results

    def run_check(self, model_instance):
        try:
            state_dict = torch.load(self.model_path, map_location=self.device)
            model_instance.load_state_dict(state_dict)
            
            results = self.evaluate(model_instance)
            
            print("\n" + "="*40)
            print("MARS MISSION: EVALUATION REPORT")
            print("="*40)
            print(f"Top-1 Accuracy:      {results['top1_accuracy']:.4f}")
            print(f"Top-3 Accuracy:      {results['top3_accuracy']:.4f} (ОСНОВНАЯ)")
            print(f"Avg Latency:         {results['avg_latency_ms']:.2f} ms")
            print(f"Model Weight:        {results['model_size_mb']:.2f} MB")
            print("="*40)

            if results['model_size_mb'] > 40:
                print("WARNING: Model exceeds 40MB limit!")
                
        except Exception as e:
            print(f"Критическая ошибка при проверке: {e}")